# Ly-$\alpha$ sFB $C_\ell(k{=}0)$ — Money Plot

Reads the output of `run_sims.py` + `compute_theory.py` and plots:
1. **Pseudo-$C_\ell$** (convolved theory vs measured) with ratio panel
2. **Deconvolved $C_\ell$** (floor-subtracted MASTER) with ratio panel

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os, sys

# Style
root = os.path.join(os.getcwd(), '..')
sys.path.insert(0, os.path.join(root, 'notebooks'))
import matplotlib_params_file

In [ ]:
# ---- CONFIGURATION ---- #
# Point these to your output files from run_sims.py and compute_theory.py
theory_file = os.path.join(root, 'results',
    'Cell_GRF_L1380_N512_Nq9797_Nl500_sims100_theory.npz')

# Fallback: look in notebooks/data for the cache produced by plot_money.py
if not os.path.exists(theory_file):
    # Try to build from old cache + plot_money.py outputs
    print(f'Theory file not found: {theory_file}')
    print('Run compute_theory.py first, or adjust the path above.')

In [ ]:
d = np.load(theory_file)
print('Keys:', list(d.keys()))

# Unpack
ells         = d['ells']
binned_ells  = d['binned_ells']
binned_raw   = d['binned_raw']
binned_std   = d['binned_std']
binned_theory = d['binned_theory']
cl_k_all     = d['cl_k_all']
cl_true      = d['cl_true']
theory_pseudo = d['theory_pseudo']
floor_cl     = float(d['floor_cl'])

ells_dec     = d['ells_dec']
meas_dec     = d['meas_dec']
meas_dec_std = d['meas_dec_std']
theory_dec   = d['theory_dec']
dec_all      = d['dec_all']

chi_eff  = float(d['chi_eff'])
Nl       = int(d['Nl'])
Nl_large = int(d['Nl_large'])
Nskew    = int(d['Nskew'])
num_sim  = int(d['num_sim'])
bias     = float(d['bias'])
NperBin  = int(d['NperBin'])

print(f'{num_sim} sims, Nl={Nl}, Nl_large={Nl_large}, Nskew={Nskew}')
print(f'chi_eff={chi_eff:.1f}, b1={bias:.4f}')

## 1. Pseudo-$C_\ell$ money plot (convolved theory)

In [ ]:
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(12, 9))
gs = GridSpec(2, 1, height_ratios=[3, 1], hspace=0.05)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)

# --- Top panel: pseudo-Cl ---
# Individual sims (light)
bins_mat = np.zeros((len(binned_ells), Nl))
for i in range(len(binned_ells)):
    lo = i * NperBin
    hi = min(lo + NperBin, Nl)
    bins_mat[i, lo:hi] = 1.0 / (hi - lo)

for i in range(min(num_sim, 100)):
    bn = bins_mat @ cl_k_all[i]
    ax1.plot(binned_ells, bn, 'k-', alpha=0.05, lw=0.3)

ax1.errorbar(binned_ells, binned_raw, yerr=binned_std,
             fmt='o', color='C0', ms=5, capsize=3, zorder=10,
             label=f'Measured mean ({num_sim} sims)')
ax1.plot(binned_ells, binned_theory, 'D--', color='C4', lw=1.5, ms=4,
         label=rf'MASTER $N_{{\ell\'}}$={Nl_large}')

ax1.set_ylabel(r'binned pseudo-$C_\ell(k{=}0)$')
ax1.legend(loc='upper right')
ax1.set_title(f'Floor-subtracted MASTER: {num_sim} sims, $N_\\ell$={Nl}, '
              f'$N_{{\\rm skew}}$={Nskew}, $b_1$={bias:.4f}')
ax1.tick_params(labelbottom=False)

# --- Bottom panel: ratio ---
ax2.axhline(1, color='k', ls='--', lw=0.8)
ax2.axhspan(0.95, 1.05, color='gray', alpha=0.15)

mask = binned_theory > 0
ratio = binned_raw[mask] / binned_theory[mask]
ratio_err = binned_std[mask] / binned_theory[mask]
ax2.errorbar(binned_ells[mask], ratio, yerr=ratio_err,
             fmt='D', color='C4', ms=4, capsize=2)

ax2.set_xlabel(r'multipole $\ell$')
ax2.set_ylabel('measured / theory')
ax2.set_ylim(0.85, 1.25)

plt.savefig('money_plot_pseudo.pdf', bbox_inches='tight')
plt.savefig('money_plot_pseudo.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'Mean ratio (excl first bin): {np.mean(ratio[1:]):.4f} ± {np.std(ratio[1:]):.4f}')

## 2. Deconvolved $C_\ell$ (floor-subtracted MASTER)

In [ ]:
fig = plt.figure(figsize=(12, 9))
gs = GridSpec(2, 1, height_ratios=[3, 1], hspace=0.05)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)

# --- Top panel ---
ax1.plot(ells_dec, theory_dec, 'k.--', lw=2,
         label=r'Theory $C_\ell = b_1^2 P(\ell/\bar\chi) / (L\bar\chi^2)$')
ax1.errorbar(ells_dec, meas_dec, yerr=meas_dec_std,
             fmt='o', color='C0', ms=5, capsize=3,
             label='Measured (deconvolved, floor-subtracted)')

ax1.set_ylabel(r'deconvolved $C_\ell(k{=}0)$')
ax1.legend()
ax1.set_title(f'Floor-subtracted MaskDeconvolution ($N_\\ell$={Nl}): {num_sim} sims')
ax1.tick_params(labelbottom=False)

# --- Bottom panel: ratio ---
ax2.axhline(1, color='k', ls='--', lw=0.8)
ax2.axhspan(0.95, 1.05, color='gray', alpha=0.15)

mask_dec = theory_dec > 0
ratio_dec = meas_dec[mask_dec] / theory_dec[mask_dec]
ratio_dec_err = meas_dec_std[mask_dec] / theory_dec[mask_dec]
ax2.errorbar(ells_dec[mask_dec], ratio_dec, yerr=ratio_dec_err,
             fmt='o', color='C0', ms=5, capsize=2)

ax2.set_xlabel(r'multipole $\ell$')
ax2.set_ylabel('measured / theory')
ax2.set_ylim(0.85, 1.35)

plt.savefig('money_plot_deconv.pdf', bbox_inches='tight')
plt.savefig('money_plot_deconv.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'Mean ratio (excl first bin): {np.mean(ratio_dec[1:]):.4f} ± {np.std(ratio_dec[1:]):.4f}')

## 3. Summary table

In [ ]:
print(f'Parameters:')
print(f'  chi_eff = {chi_eff:.1f} Mpc/h')
print(f'  bias    = {bias:.4f}')
print(f'  Nskew   = {Nskew}')
print(f'  Nl      = {Nl}, Nl_large = {Nl_large}')
print(f'  floor_cl = {floor_cl:.4e}')
print()

print(f'{"ell":>6s} {"pseudo ratio":>14s} {"deconv ratio":>14s}')
print('-' * 36)

for i in range(len(binned_ells)):
    e = binned_ells[i]
    rp = binned_raw[i] / binned_theory[i] if binned_theory[i] > 0 else np.nan
    rd = meas_dec[i] / theory_dec[i] if i < len(theory_dec) and theory_dec[i] > 0 else np.nan
    print(f'{e:6.0f} {rp:14.4f} {rd:14.4f}')